# CIP Extraction Pipeline

Run each cell in order. Review output at each step before continuing.

**Before you start:** Add your `ANTHROPIC_API_KEY` to Colab Secrets (lock icon in the left sidebar).

In [ ]:
#@title Step 0 — Setup (run once per session)
import subprocess, sys, os

def _pip(pkg):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"pip install failed for: {pkg}")
        print(result.stderr[-800:])
        raise RuntimeError(f"pip install failed: {pkg}")

from google.colab import userdata, drive

drive.mount('/content/drive', force_remount=False)

try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("API key loaded.")
except Exception:
    print("NOTE: Add ANTHROPIC_API_KEY in Colab Secrets (lock icon, left sidebar).")

print("Installing dependencies (first run takes ~60 s)...")
for _p in ['httpx', 'pdfplumber', 'openpyxl', 'pandas', 'beautifulsoup4']:
    _pip(_p)
_CIP_MODULES = {"llm": "\"\"\"\nThin httpx wrapper around the Anthropic Messages API.\nBypasses the anthropic SDK (pydantic 3.14 incompatibility).\n\nUsage:\n    from cip_tools.llm import ask, ask_json\n\n    text = ask(\"claude-sonnet-4-6\", system=\"...\", user=\"...\")\n    obj  = ask_json(\"claude-sonnet-4-6\", system=\"...\", user=\"...\")  # parses first JSON block\n\"\"\"\n\nimport json\nimport os\nimport re\nimport time\n\nimport httpx\n\nANTHROPIC_API_URL = \"https://api.anthropic.com/v1/messages\"\nANTHROPIC_VERSION = \"2023-06-01\"\n\n# Model aliases\nHAIKU  = \"claude-haiku-4-5-20251001\"\nSONNET = \"claude-sonnet-4-6\"\nOPUS   = \"claude-opus-4-7\"\n\nDEFAULT_MODEL = SONNET\n\n\ndef _api_key() -> str:\n    key = os.environ.get(\"ANTHROPIC_API_KEY\") or os.environ.get(\"CLAUDE_API\")\n    if not key:\n        raise RuntimeError(\n            \"No Anthropic API key found. Set ANTHROPIC_API_KEY or CLAUDE_API \"\n            \"environment variable, or run via: doppler run -- python -m cip_tools ...\"\n        )\n    return key\n\n\ndef ask(\n    user: str,\n    *,\n    system: str = \"\",\n    model: str = DEFAULT_MODEL,\n    max_tokens: int = 4096,\n    temperature: float = 0.2,\n    retries: int = 3,\n) -> str:\n    \"\"\"Send a message to Claude and return the text response.\"\"\"\n    headers = {\n        \"x-api-key\": _api_key(),\n        \"anthropic-version\": ANTHROPIC_VERSION,\n        \"content-type\": \"application/json\",\n    }\n    payload: dict = {\n        \"model\": model,\n        \"max_tokens\": max_tokens,\n        \"temperature\": temperature,\n        \"messages\": [{\"role\": \"user\", \"content\": user}],\n    }\n    if system:\n        payload[\"system\"] = system\n\n    for attempt in range(retries):\n        try:\n            resp = httpx.post(\n                ANTHROPIC_API_URL,\n                headers=headers,\n                json=payload,\n                timeout=120.0,\n            )\n            resp.raise_for_status()\n            data = resp.json()\n            return data[\"content\"][0][\"text\"]\n        except httpx.HTTPStatusError as e:\n            if e.response.status_code == 529 and attempt < retries - 1:\n                time.sleep(30 * (attempt + 1))\n                continue\n            raise\n    raise RuntimeError(\"Exhausted retries\")\n\n\ndef ask_json(\n    user: str,\n    *,\n    system: str = \"\",\n    model: str = DEFAULT_MODEL,\n    max_tokens: int = 4096,\n) -> dict | list:\n    \"\"\"\n    Like ask() but extracts and parses the first JSON block from the response.\n    Raises ValueError if no valid JSON is found.\n    \"\"\"\n    raw = ask(user, system=system, model=model, max_tokens=max_tokens)\n\n    # Try direct parse first\n    try:\n        return json.loads(raw)\n    except json.JSONDecodeError:\n        pass\n\n    # Look for ```json ... ``` block\n    m = re.search(r\"```json\\s*\\n([\\s\\S]*?)\\n```\", raw)\n    if m:\n        return json.loads(m.group(1))\n\n    # Look for any ``` ... ``` block\n    m = re.search(r\"```\\s*\\n([\\s\\S]*?)\\n```\", raw)\n    if m:\n        return json.loads(m.group(1))\n\n    # Look for bare { ... } or [ ... ] block\n    m = re.search(r\"(\\{[\\s\\S]*\\}|\\[[\\s\\S]*\\])\", raw)\n    if m:\n        return json.loads(m.group(1))\n\n    raise ValueError(f\"No JSON found in LLM response:\\n{raw[:500]}\")\n", "schema": "\"\"\"\nStandard output schema for _final.csv files.\n\nFINAL_COLS is the ordered list of columns in every _final.csv.\ngate_cell3() and gate_cell7() are validation checks \u2014 raise ValueError\nwith a human-readable message on failure. Warnings are printed.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom typing import Any\n\n# ---------------------------------------------------------------------------\n# Standard schema\n# ---------------------------------------------------------------------------\n\nFINAL_COLS = [\n    \"Project_Index\",\n    \"Project_Title\",\n    \"Project_Number_Primary\",\n    \"Project_Number_All_JSON\",\n    \"Client_Department\",\n    \"Total_Project_Budget\",\n    \"Fund_Names_JSON\",\n    \"Fund_Budgets_JSON\",\n    \"Yearly_Costs_By_Category_JSON\",\n    \"Description\",\n    \"Scope\",\n]\n\nREQUIRED_COLS = {\n    \"Project_Index\",\n    \"Project_Title\",\n    \"Total_Project_Budget\",\n}\n\n# ---------------------------------------------------------------------------\n# Cell 3 gate \u2014 check at ingestion, before any LLM calls\n# ---------------------------------------------------------------------------\n\ndef gate_cell3(rows: list[dict[str, Any]], source_label: str = \"\", fmt: str | None = None) -> list[str]:\n    \"\"\"\n    Run critical ingestion checks. Returns a list of warning strings.\n    Raises ValueError for fatal problems (stop everything).\n    PDFs are document-oriented; row checks don't apply \u2014 Claude works from raw text.\n    \"\"\"\n    warnings = []\n    label = f\"[{source_label}] \" if source_label else \"\"\n\n    if fmt == \"pdf\":\n        if not rows:\n            warnings.append(f\"{label}PDF: no tables detected \u2014 Claude will work from page text.\")\n        return warnings\n\n    if not rows:\n        raise ValueError(f\"{label}Empty file \u2014 zero rows extracted.\")\n\n    n = len(rows)\n    if n < 3:\n        raise ValueError(f\"{label}Only {n} row(s) \u2014 likely a header-detection failure.\")\n\n    if n < 10:\n        warnings.append(f\"{label}Only {n} rows \u2014 unusually small CIP, verify source.\")\n\n    # Check that we have at least one name-like column\n    cols = list(rows[0].keys())\n    name_cols = [c for c in cols if re.search(r'title|name|project|description', c, re.I)]\n    if not name_cols:\n        raise ValueError(\n            f\"{label}No column with 'title', 'name', 'project', or 'description' found. \"\n            f\"Header detection may have failed. Columns: {cols[:10]}\"\n        )\n\n    # Check that we have at least one dollar/value column\n    value_cols = [c for c in cols if re.search(r'budget|cost|amount|total|value|fund|\\$|fy\\d{4}|\\d{4}', c, re.I)]\n    if not value_cols:\n        raise ValueError(\n            f\"{label}No monetary column found. Cannot confirm this is a CIP. \"\n            f\"Columns: {cols[:10]}\"\n        )\n\n    # Check for all-numeric column names (header shifted by one row)\n    numeric_col_names = [c for c in cols if re.fullmatch(r'\\d+', str(c).strip())]\n    if len(numeric_col_names) > 2:\n        raise ValueError(\n            f\"{label}{len(numeric_col_names)} all-numeric column names \u2014 header row likely not detected. \"\n            f\"Sample: {numeric_col_names[:5]}\"\n        )\n\n    # Blank name rate\n    first_name_col = name_cols[0]\n    blank_names = sum(1 for r in rows if not str(r.get(first_name_col, \"\")).strip())\n    blank_rate = blank_names / n\n    if blank_rate > 0.5:\n        raise ValueError(\n            f\"{label}{blank_rate:.0%} of rows have blank '{first_name_col}' \u2014 \"\n            \"extraction failure or wrong column.\"\n        )\n    if blank_rate > 0.1:\n        warnings.append(\n            f\"{label}{blank_rate:.0%} of rows have blank '{first_name_col}' \u2014 \"\n            \"check for subtotals or header rows mixed in.\"\n        )\n\n    # Duplicate row check\n    row_reprs = [str(sorted(r.items())) for r in rows]\n    n_dupes = n - len(set(row_reprs))\n    if n_dupes / n > 0.2:\n        warnings.append(\n            f\"{label}{n_dupes}/{n} duplicate rows ({n_dupes/n:.0%}) \u2014 \"\n            \"possible double-extraction or section headers repeated.\"\n        )\n\n    return warnings\n\n\n# ---------------------------------------------------------------------------\n# Cell 7 gate \u2014 check after transform, before LLM calls\n# ---------------------------------------------------------------------------\n\ndef gate_cell7(rows: list[dict[str, Any]], source_label: str = \"\") -> list[str]:\n    \"\"\"\n    Run post-transform checks on standard schema rows.\n    Returns warnings; raises ValueError for fatal issues.\n    \"\"\"\n    warnings = []\n    label = f\"[{source_label}] \" if source_label else \"\"\n    n = len(rows)\n\n    # Project_Index must be present and sequential\n    indices = [r.get(\"Project_Index\") for r in rows]\n    none_idx = sum(1 for i in indices if i is None or str(i).strip() == \"\")\n    if none_idx > 0:\n        raise ValueError(f\"{label}{none_idx} rows missing Project_Index.\")\n\n    idx_vals = [int(i) for i in indices if str(i).strip().isdigit()]\n    if len(idx_vals) == n:\n        if sorted(idx_vals) != list(range(1, n + 1)) and sorted(idx_vals) != list(range(0, n)):\n            warnings.append(f\"{label}Project_Index values are not sequential 1..{n}.\")\n\n    # Project_Title uniqueness and fill\n    titles = [str(r.get(\"Project_Title\", \"\")).strip() for r in rows]\n    blank_titles = sum(1 for t in titles if not t)\n    if blank_titles > 0:\n        raise ValueError(f\"{label}{blank_titles} rows have blank Project_Title.\")\n    dupes = n - len(set(titles))\n    if dupes > 0:\n        warnings.append(\n            f\"{label}{dupes} duplicate Project_Title values \u2014 \"\n            \"verify these are truly distinct projects.\"\n        )\n\n    # All-zero budget rows\n    zero_budget = sum(\n        1 for r in rows\n        if _to_num(r.get(\"Total_Project_Budget\", 0)) == 0\n    )\n    if zero_budget / n > 0.3:\n        warnings.append(\n            f\"{label}{zero_budget}/{n} rows have zero Total_Project_Budget \u2014 \"\n            \"check for CP (Continuing Program) rows or extraction gaps.\"\n        )\n\n    # Year-sum vs total consistency (when yearly JSON is present)\n    mismatches = 0\n    for r in rows:\n        yearly = _parse_yearly(r.get(\"Yearly_Costs_By_Category_JSON\", \"\"))\n        total = _to_num(r.get(\"Total_Project_Budget\", 0))\n        if yearly and total > 0:\n            year_sum = sum(yearly.values())\n            if year_sum > 0 and abs(year_sum - total) / total > 0.05:\n                mismatches += 1\n    if mismatches > 0:\n        warnings.append(\n            f\"{label}{mismatches} rows where yearly sum differs >5% from Total_Project_Budget.\"\n        )\n\n    return warnings\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef _to_num(v: Any) -> float:\n    if v is None:\n        return 0.0\n    try:\n        return float(str(v).replace(\",\", \"\").replace(\"$\", \"\").strip() or 0)\n    except ValueError:\n        return 0.0\n\n\ndef _parse_yearly(v: Any) -> dict[str, float]:\n    \"\"\"Return {year_str: total_amount} from Yearly_Costs_By_Category_JSON.\"\"\"\n    if not v:\n        return {}\n    try:\n        obj = json.loads(v) if isinstance(v, str) else v\n    except (json.JSONDecodeError, TypeError):\n        return {}\n\n    if isinstance(obj, dict):\n        return {k: _to_num(val) for k, val in obj.items()}\n\n    if isinstance(obj, list):\n        # List-of-dicts format: [{\"year\": 2026, \"cn\": 100, \"federal\": 0, ...}]\n        result = {}\n        for item in obj:\n            if isinstance(item, dict):\n                yr = str(item.get(\"year\", \"\"))\n                total = sum(_to_num(v) for k, v in item.items() if k != \"year\")\n                if yr:\n                    result[yr] = result.get(yr, 0) + total\n        return result\n\n    return {}\n", "ingest": "\"\"\"\nStep 1: Load and normalize a CIP source file.\n\nSupports: .csv, .xlsx, .xls, .pdf, .html/.htm\nReturns: (raw_text, sample_rows, metadata)\n\nraw_text    - normalized text representation for LLM analysis\nsample_rows - list of dicts (first 10 rows) for LLM and quick checks\nmetadata    - dict: {format, n_cols, estimated_rows, sheet_name, ...}\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport io\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\n# Common mojibake sequences (UTF-8 decoded as latin-1) -> correct chars\nMOJIBAKE_MAP = {\n    \"a\u20ac\u201c\": \"--\",    # en-dash mojibake\n    \"a\u20ac\u2122\": \"'\",     # right single quote mojibake\n    \"a\u20ac\u0153\": '\"',     # left double quote mojibake\n    \"a\u20ac\\x9d\": '\"',       # right double quote mojibake\n    \"A\u00b0\": \"\u00b0\",      # degree sign mojibake\n    \"A\u00e9\": \"\u00e9\",      # e acute mojibake\n    \"\u00c2\u00a0\": \" \",      # non-breaking space mojibake\n    \"\u2013\": \"--\",            # en-dash\n    \"\u2014\": \"--\",            # em-dash\n    \"\u2018\": \"'\",             # left single quote\n    \"\u2019\": \"'\",             # right single quote\n    \"\u201c\": '\"',             # left double quote\n    \"\u201d\": '\"',             # right double quote\n    \"\u00a0\": \" \",             # non-breaking space\n}\n\n\n# Patterns that definitively indicate the unit of raw dollar values in the document.\n# Checked against column headers AND free text (first ~8 000 chars).\n_DOLLAR_UNIT_PATTERNS: list[tuple[int, list[str]]] = [\n    (1_000_000, [\n        r\"\\bin\\s+millions\\b\",\n        r\"\\$\\s*millions?\\b\",\n        r\"\\(in\\s+\\$\\s*millions?\\)\",\n        r\"\\(\\$\\s*000,?000s?\\)\",\n        r\"amounts?\\s+in\\s+millions\",\n        r\"\\$\\s*000,?000\",\n    ]),\n    (1_000, [\n        r\"\\bin\\s+thousands\\b\",\n        r\"\\$\\s*thousands?\\b\",\n        r\"\\(in\\s+thousands\\)\",\n        r\"\\(\\$\\s*000s?\\)\",\n        r\"\\(\\$000\\)\",\n        r\"amounts?\\s+in\\s+thousands\",\n        r\"\\$\\s*000\\b\",\n        r\"\\(000s?\\)\",\n        r\"thousands\\s+of\\s+dollars\",\n    ]),\n]\n\n\ndef _detect_dollar_unit(text: str, columns: list[str] | None = None) -> int:\n    \"\"\"Return 1, 1000, or 1000000 by scanning column names and document text.\"\"\"\n    haystack = \" \".join(columns or []) + \" \" + text[:8000]\n    low = haystack.lower()\n    for unit, patterns in _DOLLAR_UNIT_PATTERNS:\n        for pat in patterns:\n            if re.search(pat, low):\n                return unit\n    return 1\n\n\ndef normalize_text(text: str) -> str:\n    for bad, good in MOJIBAKE_MAP.items():\n        text = text.replace(bad, good)\n    # Collapse runs of whitespace except newlines\n    text = re.sub(r\"[ \\t]+\", \" \", text)\n    text = re.sub(r\"\\n{3,}\", \"\\n\\n\", text)\n    return text.strip()\n\n\ndef _try_encodings(path: Path) -> str:\n    for enc in (\"utf-8-sig\", \"utf-8\", \"latin-1\", \"cp1252\"):\n        try:\n            return path.read_text(encoding=enc)\n        except UnicodeDecodeError:\n            continue\n    return path.read_text(encoding=\"utf-8\", errors=\"replace\")\n\n\ndef load_csv(path: Path) -> tuple[str, list[dict], dict]:\n    raw = _try_encodings(path)\n    text = normalize_text(raw)\n    lines = text.splitlines()\n\n    rows = []\n    try:\n        reader = csv.DictReader(io.StringIO(text))\n        rows = [dict(r) for r in reader]\n    except Exception:\n        pass\n\n    n_rows = len(rows)\n    n_cols = len(rows[0]) if rows else (len(lines[0].split(\",\")) if lines else 0)\n\n    cols = list(rows[0].keys()) if rows else []\n    dollar_unit = _detect_dollar_unit(text, cols)\n    if dollar_unit != 1:\n        print(f\"  Dollar unit detected: x{dollar_unit} (values expressed in {'thousands' if dollar_unit == 1000 else 'millions'})\")\n    return (\n        \"\\n\".join(lines[:60]),\n        rows[:10],\n        {\"format\": \"csv\", \"n_cols\": n_cols, \"estimated_rows\": n_rows, \"dollar_unit_hint\": dollar_unit},\n    )\n\n\ndef load_excel(path: Path) -> tuple[str, list[dict], dict]:\n    import pandas as pd\n\n    # Use pandas for full read\n    try:\n        df = pd.read_excel(path, sheet_name=0, header=None)\n    except Exception as e:\n        return (str(e), [], {\"format\": \"xlsx\", \"error\": str(e)})\n\n    # Auto-detect header row (first row with >50% non-null, mostly string values)\n    header_row = 0\n    for i, row in df.iterrows():\n        non_null = row.notna().sum()\n        if non_null >= max(3, len(df.columns) * 0.3):\n            str_vals = sum(1 for v in row if isinstance(v, str) and v.strip())\n            if str_vals >= non_null * 0.5:\n                header_row = i\n                break\n\n    df2 = pd.read_excel(path, sheet_name=0, header=header_row)\n    df2 = df2.dropna(how=\"all\").dropna(axis=1, how=\"all\")\n\n    rows = df2.head(10).to_dict(\"records\")\n    rows = [{str(k): (\"\" if str(v) == \"nan\" else str(v)) for k, v in r.items()} for r in rows]\n\n    col_names = list(df2.columns)\n    preview_lines = [\" | \".join(str(c) for c in col_names)]\n    for _, row in df2.head(5).iterrows():\n        preview_lines.append(\" | \".join(str(v)[:40] for v in row))\n\n    dollar_unit = _detect_dollar_unit(\"\\n\".join(preview_lines), [str(c) for c in col_names])\n    if dollar_unit != 1:\n        print(f\"  Dollar unit detected: x{dollar_unit} (values expressed in {'thousands' if dollar_unit == 1000 else 'millions'})\")\n    return (\n        \"\\n\".join(preview_lines),\n        rows,\n        {\n            \"format\": \"xlsx\",\n            \"n_cols\": len(col_names),\n            \"estimated_rows\": len(df2),\n            \"header_row\": header_row,\n            \"columns\": col_names,\n            \"dollar_unit_hint\": dollar_unit,\n        },\n    )\n\n\n_CIP_TRIGGERS = [\n    \"capital improvement\",\n    \"capital improvements program\",\n    \" cip \",\n    \"cip project\",\n    \"cip sheet\",\n]\n\n# Strong body triggers: appear on actual project-data pages, rarely in a TOC\n_CIP_STRONG_TRIGGERS = [\n    \"recommended capital\",\n    \"capital projects detail\",\n    \"project number\",\n    \"funding source\",\n    \"total project cost\",\n    \"annual action plan\",\n    \"unfunded capital\",\n    \"cip detail\",\n    \"project listing\",\n]\n\n\ndef _is_toc_page(text: str) -> bool:\n    low = text.lower()\n    if \"table of contents\" in low or \"contents\\n\" in low:\n        return True\n    dots = text.count(\"......\")\n    lines = text.count(\"\\n\") or 1\n    return dots / lines > 0.3\n\n\ndef _toc_cip_page(all_pages: list[tuple[int, str]]) -> int | None:\n    \"\"\"Scan TOC pages and extract the document page number where the CIP chapter starts.\n\n    TOC lines look like:  \"Capital Improvement Program ........... 197\"\n    We parse the trailing integer and convert from 1-based doc page to 0-based index.\n    Returns the 0-based index into all_pages, or None if not found.\n    \"\"\"\n    for _idx, (_pnum, text) in enumerate(all_pages[:30]):\n        if not _is_toc_page(text):\n            continue\n        for line in text.splitlines():\n            low = line.lower()\n            if not any(t in low for t in _CIP_TRIGGERS):\n                continue\n            # Find trailing page number: last run of digits on the line\n            m = re.search(r'(\\d+)\\s*$', line.strip())\n            if not m:\n                continue\n            doc_page = int(m.group(1))  # 1-based page number from TOC\n            # Find the matching index in all_pages (page numbers are 0-based there)\n            for idx, (pnum, _) in enumerate(all_pages):\n                if pnum + 1 == doc_page:\n                    return idx\n            # If exact match not found (roman numeral offset), search nearby\n            for idx, (pnum, _) in enumerate(all_pages):\n                if abs((pnum + 1) - doc_page) <= 5:\n                    return idx\n    return None\n\n\ndef load_pdf(path: Path) -> tuple[str, list[dict], dict]:\n    \"\"\"Extract text from a CIP PDF.\n\n    Scans the FULL document to find the CIP chapter, then returns that\n    section's text so Claude can analyze the actual project data.\n\n    Detection priority:\n    1. TOC page number reference (most reliable for large budget PDFs)\n    2. First page with a strong body trigger (project-data keywords)\n    3. First non-TOC page with a weak CIP trigger\n    4. Fall back to start of document\n    \"\"\"\n    try:\n        import pdfplumber\n    except ImportError:\n        return (\"[pdfplumber not installed]\", [], {\"format\": \"pdf\"})\n\n    # --- Pass 1: extract text from every page ---\n    all_pages: list[tuple[int, str]] = []\n    with pdfplumber.open(path) as pdf:\n        total_pages = len(pdf.pages)\n        print(f\"  Scanning {total_pages} pages for CIP section...\")\n        for i, pg in enumerate(pdf.pages):\n            text = pg.extract_text() or \"\"\n            if text.strip():\n                all_pages.append((i, normalize_text(text)))\n\n    # --- Pass 2: find where the CIP chapter starts ---\n    toc_start = _toc_cip_page(all_pages)\n\n    strong_start = None\n    weak_start = None\n    if toc_start is None:\n        for idx, (page_num, text) in enumerate(all_pages):\n            low = text.lower()\n            if strong_start is None and any(t in low for t in _CIP_STRONG_TRIGGERS):\n                strong_start = idx\n                break\n            if weak_start is None and any(t in low for t in _CIP_TRIGGERS):\n                if not _is_toc_page(text):\n                    weak_start = idx\n\n    if toc_start is not None:\n        cip_start, method = toc_start, \"TOC\"\n    elif strong_start is not None:\n        cip_start, method = strong_start, \"strong trigger\"\n    elif weak_start is not None:\n        cip_start, method = weak_start, \"weak trigger\"\n    else:\n        cip_start, method = 0, \"fallback\"\n\n    # Collect up to 100 pages of CIP content\n    cip_pages = all_pages[cip_start: cip_start + 100]\n    page_num_start = cip_pages[0][0] + 1 if cip_pages else 1\n\n    print(f\"  CIP section found at page {page_num_start} ({method}, \"\n          f\"{len(cip_pages)} pages collected).\")\n\n    full_cip_text = \"\\n\\n\".join(\n        f\"[page {p+1}]\\n{t}\" for p, t in cip_pages\n    )\n\n    # --- Pass 3: find per-project detail pages ---\n    # Some CIPs have a summary chapter (what the TOC pointed to) plus a separate Appendix of\n    # per-project sheets located ELSEWHERE in the document.  Scan the whole PDF for those.\n    _PROJECT_PAGE_SIGNALS = [\n        \"project number\", \"project no\", \"project id\",\n        \"department:\", \"funding source:\", \"description:\",\n        \"scope:\", \"total cost:\", \"project title\",\n        \"project name\", \"project manager\",\n    ]\n    _SUMMARY_PAGE_SIGNALS = [\n        \"table of contents\", \"executive summary\", \"by department\", \"by fund\",\n        \"total uses\", \"total sources\", \"not recommended\",\n        \"grand total\", \"five-year summary\", \"5-year summary\",\n    ]\n\n    cip_page_indices = {p for p, _ in cip_pages}\n\n    # Find detail pages inside the collected CIP section\n    project_detail_start = 0\n    for i, (_, text) in enumerate(cip_pages):\n        low = text.lower()\n        sig = sum(1 for s in _PROJECT_PAGE_SIGNALS if s in low)\n        summ = sum(1 for s in _SUMMARY_PAGE_SIGNALS if s in low)\n        if (sig >= 3 and summ == 0) or sig >= 5:\n            project_detail_start = i\n            break\n\n    # Whole-document sweep: find detail pages NOT already in the CIP section.\n    # Handles docs where Appendix A lives in a different chapter than the CIP summary.\n    extra_detail_pages: list[tuple[int, str]] = []\n    for page_num, text in all_pages:\n        if page_num in cip_page_indices:\n            continue\n        low = text.lower()\n        sig = sum(1 for s in _PROJECT_PAGE_SIGNALS if s in low)\n        summ = sum(1 for s in _SUMMARY_PAGE_SIGNALS if s in low)\n        if sig >= 4 and summ == 0:\n            extra_detail_pages.append((page_num, text))\n\n    if extra_detail_pages:\n        extra_text = \"\\n\\n\".join(f\"[page {p+1}]\\n{t}\" for p, t in extra_detail_pages[:50])\n        full_cip_text = extra_text + \"\\n\\n\" + full_cip_text\n        print(f\"  Found {len(extra_detail_pages)} project-detail pages outside CIP section \"\n              f\"(pages {extra_detail_pages[0][0]+1}\u2013{extra_detail_pages[-1][0]+1}).\")\n\n    # 5-page sample: prefer extra_detail_pages if they exist, else fall back to cip_pages\n    if extra_detail_pages:\n        sample_pages = extra_detail_pages[:5]\n    else:\n        sample_pages = cip_pages[project_detail_start: project_detail_start + 5]\n\n    project_sample_text = \"\\n\\n\".join(\n        f\"[page {p+1}]\\n{t}\" for p, t in sample_pages\n    )\n    sample_page_start = sample_pages[0][0] + 1 if sample_pages else page_num_start\n    print(f\"  Project detail sample: pages {sample_page_start}-\"\n          f\"{sample_pages[-1][0]+1 if sample_pages else sample_page_start} \"\n          f\"({len(project_sample_text)} chars).\")\n\n    # Try table extraction on the sample project pages for sample_rows\n    sample_rows: list[dict] = []\n    with pdfplumber.open(path) as pdf:\n        for p, _ in sample_pages[:5]:\n            tables = pdf.pages[p].extract_tables()\n            for tbl in tables[:1]:\n                if tbl and len(tbl) > 1:\n                    headers = [str(c or \"\").strip() for c in tbl[0]]\n                    for row in tbl[1:6]:\n                        d = {headers[j]: str(v or \"\").strip()\n                             for j, v in enumerate(row) if j < len(headers)}\n                        sample_rows.append(d)\n            if len(sample_rows) >= 10:\n                break\n\n    dollar_unit = _detect_dollar_unit(full_cip_text[:8000])\n    if dollar_unit != 1:\n        print(f\"  Dollar unit detected: x{dollar_unit} (values expressed in {'thousands' if dollar_unit == 1000 else 'millions'})\")\n    return (\n        project_sample_text[:12000],   # raw_text shown in Step 1 = actual project pages\n        sample_rows[:10],\n        {\n            \"format\": \"pdf\",\n            \"total_pages\": total_pages,\n            \"cip_section_start_page\": page_num_start,\n            \"cip_pages_collected\": len(cip_pages),\n            \"project_sample_start_page\": sample_page_start,\n            \"n_cols\": len(sample_rows[0]) if sample_rows else 0,\n            \"estimated_rows\": len(sample_rows),\n            \"full_cip_text\": full_cip_text,        # all CIP pages, for the extraction script\n            \"project_sample_text\": project_sample_text,  # 5 project pages, for analysis prompts\n            \"dollar_unit_hint\": dollar_unit,\n        },\n    )\n\n\ndef load_html(path: Path) -> tuple[str, list[dict], dict]:\n    try:\n        from bs4 import BeautifulSoup\n    except ImportError:\n        return (\"[beautifulsoup4 not installed]\", [], {\"format\": \"html\"})\n\n    raw = _try_encodings(path)\n    soup = BeautifulSoup(raw, \"html.parser\")\n    tables = soup.find_all(\"table\")\n\n    if not tables:\n        text = normalize_text(soup.get_text(\" \", strip=True))\n        return text[:3000], [], {\"format\": \"html\", \"n_tables\": 0}\n\n    tbl = tables[0]\n    header_row = tbl.find(\"tr\")\n    headers = [th.get_text(strip=True) for th in header_row.find_all([\"th\", \"td\"])] if header_row else []\n\n    rows = []\n    for tr in tbl.find_all(\"tr\")[1:11]:\n        cells = [td.get_text(strip=True) for td in tr.find_all([\"td\", \"th\"])]\n        if cells:\n            row = {headers[i] if i < len(headers) else str(i): v for i, v in enumerate(cells)}\n            rows.append(row)\n\n    preview = [\" | \".join(headers)] + [\" | \".join(str(v)[:30] for v in r.values()) for r in rows[:4]]\n\n    preview_text = \"\\n\".join(preview)\n    dollar_unit = _detect_dollar_unit(preview_text, headers)\n    if dollar_unit != 1:\n        print(f\"  Dollar unit detected: x{dollar_unit} (values expressed in {'thousands' if dollar_unit == 1000 else 'millions'})\")\n    return (\n        preview_text,\n        rows,\n        {\"format\": \"html\", \"n_tables\": len(tables), \"n_cols\": len(headers), \"dollar_unit_hint\": dollar_unit},\n    )\n\n\ndef load(path: str | Path) -> tuple[str, list[dict], dict]:\n    \"\"\"\n    Dispatch to the right loader based on file extension.\n    Returns (raw_text, sample_rows, metadata).\n    \"\"\"\n    p = Path(path)\n    if not p.exists():\n        raise FileNotFoundError(f\"Source file not found: {p}\")\n\n    suffix = p.suffix.lower()\n    if suffix == \".csv\":\n        return load_csv(p)\n    if suffix in (\".xlsx\", \".xls\"):\n        return load_excel(p)\n    if suffix == \".pdf\":\n        return load_pdf(p)\n    if suffix in (\".html\", \".htm\"):\n        return load_html(p)\n\n    # Try CSV as fallback for unknown extensions\n    return load_csv(p)\n", "understand": "\"\"\"\nStep 2: Ask Claude to analyze the structure of a CIP source file.\n\nSends N evenly-spaced samples from the document in a single call.\nN scales with document size: max(1, min(6, round(cip_pages / 15))).\nFor PDFs, uses the full CIP section text stored in metadata by ingest.load_pdf().\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Any\n\nfrom .llm import ask_json, SONNET\n\nSYSTEM = \"\"\"You are a capital infrastructure plan (CIP) data extraction expert.\nYou analyze government CIP documents \u2014 PDFs, spreadsheets, CSVs \u2014 and describe their structure\nso a Python script can extract one row per project into a standard schema.\n\nStandard output schema (every _final.csv must contain these columns):\n  Project_Index              integer, 1-based row number\n  Project_Title              string, human-readable project name\n  Project_Number_Primary     string, the main ID/code for this project (e.g. \"HW-123\")\n  Project_Number_All_JSON    JSON array of all codes found (can be [\"HW-123\"])\n  Client_Department          string, owning department/agency\n  Total_Project_Budget       number, total dollar amount (all years, all funds combined)\n  Fund_Names_JSON            JSON array of funding source names (e.g. [\"City\", \"Federal\"])\n  Fund_Budgets_JSON          JSON array of amounts matching Fund_Names_JSON\n  Yearly_Costs_By_Category_JSON  JSON object {\"2026\": amount, \"2027\": amount, ...}\n  Description                string, free-text description of the project\n  Scope                      string, scope, location, or additional detail\n\nReturn ONLY a JSON object \u2014 no prose, no markdown fences.\"\"\"\n\nSCHEMA_JSON = \"\"\"{\n  \"format_type\": \"wide_excel|csv_tabular|pdf_table|long_csv|web_html|other\",\n  \"header_row_index\": null_or_integer,\n  \"project_name_col\": \"column name or null\",\n  \"project_id_col\": \"column name or null\",\n  \"department_col\": \"column name or null\",\n  \"description_col\": \"column name or null\",\n  \"scope_col\": \"column name or null\",\n  \"total_budget_col\": \"column name or null\",\n  \"year_cols\": [\"col_name_2026\", \"col_name_2027\"],\n  \"fund_cols\": [{\"col\": \"col_name\", \"fund_name\": \"City Notes\", \"year\": null_or_year}],\n  \"project_count_estimate\": integer,\n  \"published_grand_total\": null_or_number,\n  \"dollar_unit\": 1,\n  \"multi_row_projects\": true_or_false,\n  \"quirks\": [\"list of unusual features\"],\n  \"extraction_approach\": \"detailed step-by-step description of how to extract rows\",\n  \"notes\": \"anything the curator should know\"\n}\"\"\"\n\nANALYSIS_TEMPLATE = \"\"\"Analyze this CIP source file and return a JSON object describing its structure.\n\n=== FILE: {filename} ===\n=== FORMAT: {fmt} ===\n=== METADATA: {metadata} ===\n\n=== SAMPLE ROWS (as parsed dicts) ===\n{sample_rows}\n\n{content_sections}\n\nFor published_grand_total: look for a summary table or total line. Return the number only (no $ or commas), or null.\nFor dollar_unit: the regex pre-scan at intake set dollar_unit_hint={dollar_unit_hint}. Treat this as strong\nevidence. Also scan column headers, footnotes, and table titles for phrases like \"(in thousands)\", \"($000s)\",\n\"amounts in thousands\", \"in millions\", \"$ millions\", \"000s omitted\". Consider typical project sizes \u2014\nif an infrastructure project shows a total of \"500\" it is almost certainly in thousands ($500,000), not $500.\nSet dollar_unit to 1 (full dollars), 1000 (thousands), or 1000000 (millions).\nFor extraction_approach: describe step-by-step how a Python script should extract one row per project.\nSet extraction_approach and other fields based on the MOST REPRESENTATIVE content section(s) you find.\n\nReturn this JSON:\n{schema}\"\"\"\n\nDEFAULTS = {\n    \"format_type\": \"unknown\",\n    \"header_row_index\": None,\n    \"project_name_col\": None,\n    \"project_id_col\": None,\n    \"department_col\": None,\n    \"description_col\": None,\n    \"scope_col\": None,\n    \"total_budget_col\": None,\n    \"year_cols\": [],\n    \"fund_cols\": [],\n    \"project_count_estimate\": 0,\n    \"published_grand_total\": None,\n    \"dollar_unit\": 1,\n    \"multi_row_projects\": False,\n    \"quirks\": [],\n    \"extraction_approach\": \"\",\n    \"notes\": \"\",\n}\n\n\ndef analyze(\n    filename: str,\n    raw_text: str,\n    sample_rows: list[dict],\n    metadata: dict,\n) -> dict[str, Any]:\n    \"\"\"Send N evenly-spaced document samples in one call. N scales with doc size.\"\"\"\n    import json\n\n    full_text = metadata.get(\"full_cip_text\") or metadata.get(\"project_sample_text\") or raw_text\n    fmt = metadata.get(\"format\", \"unknown\")\n\n    # Scale N to document size: 1 sample per ~15 CIP pages, capped 1\u20136\n    cip_pages = metadata.get(\"cip_pages_collected\", 0)\n    if cip_pages == 0:\n        # Non-PDF: single sample is sufficient\n        n_samples = 1\n    else:\n        n_samples = max(1, min(6, round(cip_pages / 15)))\n\n    # Budget ~30 000 chars total across all samples\n    sample_chars = min(30000 // n_samples, 8000)\n    total_len = len(full_text)\n\n    # Build evenly-spaced start positions (0%, 1/(n-1)%, ..., 100% - 1 window)\n    if n_samples == 1:\n        positions = [0]\n    else:\n        positions = [\n            int(i * (total_len - sample_chars) / (n_samples - 1))\n            for i in range(n_samples)\n        ]\n        positions = [max(0, min(p, total_len - sample_chars)) for p in positions]\n\n    # Extract samples and label them\n    content_sections = []\n    for i, start in enumerate(positions):\n        chunk = full_text[start: start + sample_chars]\n        if not chunk.strip():\n            continue\n        pct = int(start / total_len * 100) if total_len else 0\n        label = f\"=== CONTENT SAMPLE {i+1}/{n_samples} (document position ~{pct}%) ===\"\n        content_sections.append(f\"{label}\\n{chunk}\")\n\n    print(f\"  Sending {len(content_sections)} sample(s) \"\n          f\"({sample_chars} chars each, {cip_pages} CIP pages)...\")\n\n    dollar_unit_hint = metadata.get(\"dollar_unit_hint\", 1)\n    user_msg = ANALYSIS_TEMPLATE.format(\n        filename=filename,\n        fmt=fmt,\n        metadata=json.dumps(\n            {k: v for k, v in metadata.items()\n             if k not in (\"full_cip_text\", \"project_sample_text\")},\n            default=str)[:400],\n        sample_rows=json.dumps(sample_rows[:10], default=str, indent=2)[:1000],\n        content_sections=\"\\n\\n\".join(content_sections),\n        schema=SCHEMA_JSON,\n        dollar_unit_hint=dollar_unit_hint,\n    )\n\n    # Pre-seed dollar_unit from the regex hint; Claude can still override with a different value\n    seed = {**DEFAULTS, \"dollar_unit\": dollar_unit_hint}\n    result = {**seed, **ask_json(user_msg, system=SYSTEM, model=SONNET, max_tokens=4096)}\n    print(f\"  Analysis complete.\")\n    return {**DEFAULTS, **result}\n\n\ndef summarize(analysis: dict[str, Any]) -> str:\n    \"\"\"Return a human-readable summary of the structure analysis.\"\"\"\n    lines = [\n        f\"Format:          {analysis['format_type']}\",\n        f\"Dollar unit:     x{analysis.get('dollar_unit', 1)} (raw values multiplied to normalize to full $)\",\n        f\"Project name:    {analysis['project_name_col']}\",\n        f\"Project ID:      {analysis['project_id_col']}\",\n        f\"Department:      {analysis['department_col']}\",\n        f\"Total budget:    {analysis['total_budget_col']}\",\n        f\"Year columns:    {analysis['year_cols']}\",\n        f\"Fund columns:    {len(analysis['fund_cols'])} found\",\n        f\"Est. projects:   {analysis['project_count_estimate']}\",\n        f\"Published total: {'${:,.0f}'.format(analysis['published_grand_total']) if analysis.get('published_grand_total') else '(not found in preview)'}\",\n        f\"Multi-row:       {analysis['multi_row_projects']}\",\n        f\"Approach:        {analysis['extraction_approach']}\",\n    ]\n    if analysis[\"quirks\"]:\n        lines.append(f\"Quirks:          {'; '.join(analysis['quirks'])}\")\n    if analysis[\"notes\"]:\n        lines.append(f\"Notes:           {analysis['notes']}\")\n    return \"\\n\".join(lines)\n", "design": "\"\"\"\nStep 3: Ask Claude to write a Python extraction script for this CIP file.\n\nThe script must:\n- Have a CONFIGURATION block at top (easy to re-use next year)\n- Accept a source file path and return a list of dicts\n- Map source columns to the standard schema\n- Handle multi-value fields (funds, years) by JSON-encoding them\n- Be runnable standalone as: python <script>.py\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .llm import ask, SONNET\nfrom .schema import FINAL_COLS\n\nSYSTEM = \"\"\"You are a senior Python developer specializing in data extraction from government documents.\nWrite clean, minimal extraction scripts. No unnecessary abstractions. No error suppression.\nInclude a CONFIGURATION block so next year's curator can update paths without reading the logic.\"\"\"\n\nUSER_TEMPLATE = \"\"\"Write a Python extraction script for this CIP source file.\n{guide_block}\n=== SOURCE FILE INFO ===\nFilename:  {filename}\nFull path: {full_path}\nFormat:    {format_type}\nApproach:  {approach}\n\n=== STRUCTURE ANALYSIS ===\n{analysis_json}\n\n=== SAMPLE ROWS (raw source) ===\n{sample_rows}\n\n=== EXTRACTED TEXT SAMPLE (first 8000 chars of CIP section) ===\n{text_sample}\n\n=== REQUIREMENTS ===\nThe script must:\n1. Start with a CONFIGURATION block (all tuneable values as module-level constants).\n   SOURCE_FILE must be set to the FULL PATH shown above (not just the filename).\n   Include: DOLLAR_UNIT = {dollar_unit}\n2. Define a run() function that reads the source file and returns list[dict]\n3. Each dict must contain EXACTLY these keys (use empty string for missing):\n   {final_cols}\n4. Project_Index must be sequential integers starting at 1\n5. Total_Project_Budget must be a numeric value (integer or float, no $ or commas)\n6. Fund_Names_JSON: JSON array of strings e.g. '[\"City\", \"Federal\", \"State\"]'\n7. Fund_Budgets_JSON: JSON array of numbers matching Fund_Names_JSON\n8. Yearly_Costs_By_Category_JSON: JSON object e.g. '{{\"2026\": 500000, \"2027\": 250000}}'\n9. If __name__ == \"__main__\": block that runs and prints a summary\n10. Handle encoding: try utf-8-sig first, fall back to latin-1\n11. DOLLAR_UNIT normalization: multiply EVERY parsed dollar amount by DOLLAR_UNIT after clean_num().\n    Example: total = clean_num(row.get('Total')) * DOLLAR_UNIT\n    This normalizes values expressed in thousands or millions to full dollars.\n\n=== DEFENSIVE CODING REQUIREMENTS (mandatory) ===\nInclude a clean_num() helper at the top of run():\n\n    def clean_num(val):\n        if val is None: return 0\n        s = str(val).strip().replace(',', '').replace('$', '').replace('O', '0')\n        s = ''.join(c for c in s if c.isdigit() or c in '.+-')\n        try: return float(s) if s else 0\n        except ValueError: return 0\n\nRules:\n- Use clean_num() for ALL numeric fields \u2014 never float() or int() directly on raw text\n- Use .get(key, '') or .get(key) or '' for ALL dict lookups \u2014 never bare dict[key]\n- Treat these as zero: '', '-', 'N/A', 'n/a', None, 'CP', '*', '**'\n- OCR often outputs the letter O where 0 is intended \u2014 clean_num() handles this\n- When iterating rows, skip silently if required fields are blank rather than raising\n- For column alignment: match columns by header name, never by fixed position index\n- Strip all whitespace from header names before using them as dict keys\n\n=== PDF-SPECIFIC REQUIREMENTS (apply when format is pdf) ===\nFor PDF sources, the script MUST use regex on the extracted text \u2014 not table extraction:\n- Use pdfplumber to extract raw text from every page: pg.extract_text() or \"\"\n- Concatenate all page texts into one string with [page N] markers between pages\n- Use re.split() or re.finditer() to split the full text into per-project blocks,\n  anchored on the regex pattern that marks the START of each new project entry\n- Within each block, use re.search() to extract individual fields by their labels\n- For fund rows, use re.findall() to collect all fund name / amount pairs\n- Never rely on fixed line numbers or character positions \u2014 OCR shifts them\n- The project boundary pattern and field label patterns must come from studying\n  the EXTRACTED TEXT SAMPLE above \u2014 use the actual text, not assumptions\n\nWrite ONLY the Python script, no explanation.\"\"\"\n\nGUIDE_TEMPLATE = \"\"\"# Extraction Guide: {agency_id}\n\n## Source File\n- **File**: `{filename}`\n- **Format**: {format_type}\n- **Estimated projects**: {project_count}\n\n## Structure Analysis\n{analysis_summary}\n\n## Extraction Script\n```python\n{script}\n```\n\n## Configuration\nKey variables to update each year:\n{config_notes}\n\n## QA Checks\n- [ ] Row count matches source project count\n- [ ] Total budget matches published grand total\n- [ ] Spot-check 3 named projects against source\n- [ ] No blank Project_Title values\n- [ ] Fund breakdown sums match Total_Project_Budget\n\n## Adaptation Notes\nNext year: update `SOURCE_FILE` path and verify year column names still match.\n\"\"\"\n\n\ndef _build_guide_block(guide_context: str, guide_mode: str) -> str:\n    \"\"\"Format the guide context section for injection into the prompt.\"\"\"\n    if not guide_context:\n        return \"\"\n    if guide_mode == \"prior_year\":\n        return (\n            \"\\n=== PRIOR-YEAR GUIDE (SAME AGENCY) ===\\n\"\n            \"This is the complete extraction guide for this same agency's previous CIP.\\n\"\n            \"The new CIP should have an identical structure \u2014 use this as your primary template.\\n\"\n            \"Update only: year ranges, SOURCE_FILE path, and any column names that changed.\\n\\n\"\n            f\"{guide_context}\\n\"\n        )\n    return (\n        \"\\n=== SIMILAR PAST EXTRACTION GUIDES ===\\n\"\n        \"These guides show how we extracted similar CIPs (same format / doc type).\\n\"\n        \"Study their CONFIGURATION constants \u2014 especially SECTION_MARKERS, LEADER_RE, and YEARS \u2014\\n\"\n        \"as starting points. Adapt them to match the NEW CIP's structure in STRUCTURE ANALYSIS.\\n\\n\"\n        f\"{guide_context}\\n\"\n    )\n\n\ndef write_script(\n    filename: str,\n    analysis: dict[str, Any],\n    sample_rows: list[dict],\n    full_path: str = \"\",\n    metadata: dict | None = None,\n    guide_context: str = \"\",\n    guide_mode: str = \"\",\n) -> str:\n    \"\"\"Ask Claude to generate the extraction script. Returns Python source code.\"\"\"\n    # For PDFs, send the focused project-page sample so Claude can write concrete regex patterns.\n    # For tabular files, an empty text_sample is fine (structure comes from column names).\n    text_sample = \"\"\n    if metadata:\n        text_sample = metadata.get(\"project_sample_text\", \"\") or metadata.get(\"full_cip_text\", \"\")\n    text_sample = text_sample[:8000]  # keep prompt manageable\n\n    user_msg = USER_TEMPLATE.format(\n        guide_block=_build_guide_block(guide_context, guide_mode),\n        filename=filename,\n        full_path=full_path or filename,\n        format_type=analysis.get(\"format_type\", \"unknown\"),\n        approach=analysis.get(\"extraction_approach\", \"\"),\n        analysis_json=json.dumps(analysis, indent=2)[:2000],\n        sample_rows=json.dumps(sample_rows[:5], default=str, indent=2)[:1500],\n        text_sample=text_sample or \"(not available \u2014 use column analysis above)\",\n        final_cols=\"\\n   \".join(FINAL_COLS),\n        dollar_unit=analysis.get(\"dollar_unit\", 1),\n    )\n    raw = ask(user_msg, system=SYSTEM, model=SONNET, max_tokens=4096).strip()\n\n    # Strip markdown fences by slicing from first/last fence lines\n    if raw.startswith(\"```\"):\n        lines = raw.splitlines()\n        lines = lines[1:]  # drop opening ```python line\n        if lines and lines[-1].strip().startswith(\"```\"):\n            lines = lines[:-1]  # drop closing ``` line\n        return \"\\n\".join(lines).strip()\n    return raw\n\n\ndef make_guide(\n    agency_id: str,\n    filename: str,\n    analysis: dict[str, Any],\n    script: str,\n    analysis_summary: str,\n) -> str:\n    \"\"\"Generate the _guide.md content.\"\"\"\n    # Pull config vars from script (lines starting with uppercase identifiers)\n    import re\n    config_lines = [\n        l.strip() for l in script.splitlines()\n        if re.match(r'^[A-Z_]+ *=', l.strip())\n    ]\n    config_notes = \"\\n\".join(f\"- `{l}`\" for l in config_lines[:10])\n\n    return GUIDE_TEMPLATE.format(\n        agency_id=agency_id,\n        filename=filename,\n        format_type=analysis.get(\"format_type\", \"unknown\"),\n        project_count=analysis.get(\"project_count_estimate\", \"?\"),\n        analysis_summary=analysis_summary,\n        script=script,\n        config_notes=config_notes or \"- Update `SOURCE_FILE` path\",\n    )\n", "guide_rag": "\"\"\"\nGuide RAG: find the best past *_guide.md examples for a new CIP.\n\nPriority:\n  1. Prior-year guide for the SAME agency (exact domain match) \u2014 sole example.\n  2. Similar guides retrieved by format + doc-type + year-span \u2014 fallback.\n\nIndex stored at:\n  /content/drive/Shareddrives/0_cip_data/extract/_guide_index.json\n\nBuild once per session via build_index(); reload cheaply with load_index().\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport re\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\n\n# ---------------------------------------------------------------------------\n# Index build\n# ---------------------------------------------------------------------------\n\ndef _extract_metadata(guide_path: str) -> dict[str, Any]:\n    \"\"\"Parse a guide file and return a metadata dict (regex only, no LLM).\"\"\"\n    try:\n        text = Path(guide_path).read_text(encoding=\"utf-8\", errors=\"replace\")\n    except OSError:\n        return {}\n\n    fname = Path(guide_path).name  # e.g. wallawallawa.gov_tip_2026-2031_guide.md\n    agency_id = fname.replace(\"_guide.md\", \"\")\n\n    # Domain: everything up to and including the first TLD segment\n    domain_m = re.match(r\"^([a-z0-9][a-z0-9.\\-]*\\.[a-z]{2,6})\", agency_id)\n    agency_domain = domain_m.group(1) if domain_m else agency_id.split(\"_\")[0]\n\n    # Doc type\n    if \"_tip_\" in agency_id or agency_id.endswith(\"_tip\"):\n        doc_type = \"tip\"\n    elif \"_cfp_\" in agency_id or agency_id.endswith(\"_cfp\"):\n        doc_type = \"cfp\"\n    else:\n        doc_type = \"cip\"\n\n    # Broad file format \u2014 what library the script uses\n    if \"pdfplumber\" in text:\n        file_format = \"pdf\"\n    elif \"read_excel\" in text or \"openpyxl\" in text:\n        file_format = \"excel\"\n    elif \"csv.DictReader\" in text or \"read_csv\" in text:\n        file_format = \"csv\"\n    else:\n        file_format = \"other\"\n\n    # Layout type \u2014 extracted from \"Format: <value>\" line in Structure Analysis section.\n    # understand.py writes this as e.g. \"Format:          pdf_table\" or \"pdf_one_per_page\".\n    # Known values: pdf_table, pdf_one_per_page, pdf_project_blocks, pdf_project_sheets,\n    #               wide_excel, csv_tabular, long_csv, web_html, other.\n    layout_m = re.search(r\"^Format:\\s+(\\S+)\", text, re.MULTILINE)\n    if layout_m:\n        format_type = layout_m.group(1)\n    else:\n        # Infer from script patterns for older guides written before layout types existed\n        if file_format == \"pdf\":\n            if re.search(r\"LEADER_RE|split.*block|split.*project|per.?page\", text, re.IGNORECASE):\n                format_type = \"pdf_one_per_page\"\n            elif re.search(r\"SECTION_MARKERS\", text):\n                format_type = \"pdf_project_blocks\"\n            else:\n                format_type = \"pdf_table\"\n        elif file_format == \"excel\":\n            format_type = \"wide_excel\"\n        elif file_format == \"csv\":\n            format_type = \"csv_tabular\"\n        else:\n            format_type = \"other\"\n\n    # Year list from YEARS = [2026, 2027, ...]\n    years: list[int] = []\n    years_m = re.search(r\"YEARS\\s*=\\s*\\[([^\\]]+)\\]\", text)\n    if years_m:\n        years = [int(y) for y in re.findall(r\"\\d{4}\", years_m.group(1))]\n    if not years:\n        # Fallback: extract from agency_id year range e.g. 2026-2031\n        yr_m = re.search(r\"(\\d{4})-(\\d{4})\", agency_id)\n        if yr_m:\n            y0, y1 = int(yr_m.group(1)), int(yr_m.group(2))\n            years = list(range(y0, y1 + 1))\n\n    # Section count from SECTION_MARKERS\n    section_m = re.search(r\"SECTION_MARKERS\\s*=\\s*\\{([^}]+)\\}\", text, re.DOTALL)\n    section_count = len(re.findall(r'\"[^\"]+\"\\s*:', section_m.group(1))) if section_m else 0\n\n    # Has a compiled leader regex?\n    has_leader_re = bool(re.search(r\"LEADER_RE\\s*=\\s*re\\.compile\", text))\n\n    # Multi-row projects \u2014 check Structure Analysis section of the guide\n    multi_row_m = re.search(r\"Multi-row:\\s+(True|False)\", text, re.IGNORECASE)\n    multi_row_projects = (multi_row_m.group(1).lower() == \"true\") if multi_row_m else False\n\n    return {\n        \"guide_path\": guide_path,\n        \"agency_id\": agency_id,\n        \"agency_domain\": agency_domain,\n        \"doc_type\": doc_type,\n        \"format_type\": format_type,   # layout type (pdf_table, pdf_one_per_page, etc.)\n        \"file_format\": file_format,   # broad file format (pdf, excel, csv, other)\n        \"years\": years,\n        \"year_min\": min(years) if years else 0,\n        \"year_max\": max(years) if years else 0,\n        \"year_count\": len(years),\n        \"section_count\": section_count,\n        \"has_leader_re\": has_leader_re,\n        \"multi_row_projects\": multi_row_projects,\n        \"char_count\": len(text),\n    }\n\n\ndef build_index(guide_root: str, index_path: str) -> dict:\n    \"\"\"Scan guide_root for *_guide.md files, extract metadata, write JSON index.\"\"\"\n    guide_root_p = Path(guide_root)\n    print(f\"Scanning {guide_root} for guide files...\")\n\n    guides = []\n    for p in sorted(guide_root_p.rglob(\"*_guide.md\")):\n        meta = _extract_metadata(str(p))\n        if meta:\n            guides.append(meta)\n\n    index = {\n        \"guides\": guides,\n        \"built_at\": datetime.now(timezone.utc).isoformat(),\n        \"guide_root\": guide_root,\n    }\n    Path(index_path).write_text(json.dumps(index, indent=2), encoding=\"utf-8\")\n    print(f\"Index built: {len(guides)} guides -> {index_path}\")\n    return index\n\n\ndef load_index(index_path: str) -> dict:\n    \"\"\"Load an existing guide index from JSON.\"\"\"\n    return json.loads(Path(index_path).read_text(encoding=\"utf-8\"))\n\n\n# ---------------------------------------------------------------------------\n# Retrieval\n# ---------------------------------------------------------------------------\n\ndef _parse_filename_features(filename: str) -> dict[str, Any]:\n    \"\"\"Extract domain, doc_type, year range from a source filename.\"\"\"\n    base = Path(filename).stem  # strip extension\n\n    domain_m = re.match(r\"^([a-z0-9][a-z0-9.\\-]*\\.[a-z]{2,6})\", base)\n    agency_domain = domain_m.group(1) if domain_m else base.split(\"_\")[0]\n\n    doc_type = \"cip\"\n    if \"_tip\" in base:\n        doc_type = \"tip\"\n    elif \"_cfp\" in base:\n        doc_type = \"cfp\"\n\n    yr_m = re.search(r\"(\\d{4})-(\\d{4})\", base)\n    year_min = int(yr_m.group(1)) if yr_m else 0\n    year_max = int(yr_m.group(2)) if yr_m else 0\n    year_count = (year_max - year_min + 1) if yr_m else 0\n\n    return {\n        \"agency_domain\": agency_domain,\n        \"doc_type\": doc_type,\n        \"year_min\": year_min,\n        \"year_max\": year_max,\n        \"year_count\": year_count,\n    }\n\n\ndef find_prior_year(filename: str, index: dict) -> dict | None:\n    \"\"\"Return the most recent prior guide for this exact agency domain, or None.\"\"\"\n    feat = _parse_filename_features(filename)\n    domain = feat[\"agency_domain\"]\n    candidates = [\n        g for g in index[\"guides\"]\n        if g[\"agency_domain\"] == domain\n        and g.get(\"year_min\", 0) < feat[\"year_min\"]  # must be an older year\n    ]\n    if not candidates:\n        return None\n    return max(candidates, key=lambda g: g.get(\"year_min\", 0))\n\n\ndef find_similar(filename: str, analysis: dict, index: dict, top_k: int = 3) -> list[dict]:\n    \"\"\"Score all indexed guides against the new CIP and return top_k.\n\n    Scoring (max 26):\n      +15  Exact layout type match (pdf_one_per_page, pdf_table, wide_excel, ...)\n      + 5  Same broad file format (pdf / excel / csv) when layout doesn't match\n      + 5  Same doc type (cip / tip / cfp)\n      + 3  Same year count\n      + 3  Same multi_row_projects flag\n    \"\"\"\n    feat = _parse_filename_features(filename)\n    layout_type = analysis.get(\"format_type\", \"\")\n\n    # Broad file format from layout type string\n    def _broad(lt: str) -> str:\n        if \"pdf\" in lt:\n            return \"pdf\"\n        if \"excel\" in lt or \"xlsx\" in lt:\n            return \"excel\"\n        if \"csv\" in lt:\n            return \"csv\"\n        return lt\n\n    broad_fmt = _broad(layout_type)\n    multi_row = analysis.get(\"multi_row_projects\", False)\n\n    year_cols = analysis.get(\"year_cols\", [])\n    analysis_year_count = len(year_cols) if year_cols else feat[\"year_count\"]\n\n    scored = []\n    for g in index[\"guides\"]:\n        score = 0\n\n        g_layout = g.get(\"format_type\", \"\")\n        g_broad = g.get(\"file_format\") or _broad(g_layout)\n\n        # Layout type is the strongest signal \u2014 same structure = same regex patterns\n        if g_layout and g_layout == layout_type:\n            score += 15\n        elif g_broad == broad_fmt:\n            score += 5  # same file format but different layout sub-type\n\n        # Doc type: TIP vs CIP is structurally significant\n        if g.get(\"doc_type\") == feat[\"doc_type\"]:\n            score += 5\n\n        # Year count: 5-year vs 6-year changes column alignment logic\n        if g.get(\"year_count\") == analysis_year_count:\n            score += 3\n\n        # Multi-row projects flag: changes row-grouping logic significantly\n        if g.get(\"multi_row_projects\") == multi_row:\n            score += 3\n\n        if score > 0:\n            scored.append((score, g))\n\n    scored.sort(key=lambda x: -x[0])\n    return [g for _, g in scored[:top_k]]\n\n\n# ---------------------------------------------------------------------------\n# Guide excerpt extraction\n# ---------------------------------------------------------------------------\n\ndef _extract_script_block(text: str) -> str:\n    \"\"\"Pull the main Python extraction script from a guide (the largest ```python block).\"\"\"\n    blocks = re.findall(r\"```python\\s*\\n([\\s\\S]*?)\\n```\", text)\n    if not blocks:\n        return \"\"\n    # The extraction script is always the longest block in the guide\n    return max(blocks, key=len).strip()\n\n\ndef _extract_config_constants(script: str) -> str:\n    \"\"\"Return the CONFIGURATION block \u2014 lines from start up to first class/dataclass/def run.\"\"\"\n    lines = script.splitlines()\n    config_lines = []\n    for line in lines:\n        # Stop when we hit the data structures / function definitions\n        if re.match(r\"^(class |def |@dataclass)\", line.strip()):\n            break\n        config_lines.append(line)\n    return \"\\n\".join(config_lines).strip()\n\n\ndef get_guide_excerpt(guide_path: str, mode: str = \"config\") -> str:\n    \"\"\"\n    Return relevant excerpt from a guide file.\n\n    mode=\"full\"   \u2014 entire guide (for prior-year exact match)\n    mode=\"config\" \u2014 just the CONFIGURATION constants from the script (for similar guides)\n    \"\"\"\n    try:\n        text = Path(guide_path).read_text(encoding=\"utf-8\", errors=\"replace\")\n    except OSError:\n        return \"\"\n\n    if mode == \"full\":\n        return text\n\n    # For similar guides: return the script config constants + first few section markers\n    script = _extract_script_block(text)\n    if not script:\n        return text[:3000]\n    config = _extract_config_constants(script)\n    return config[:3000]  # cap per guide so total stays manageable\n\n\n# ---------------------------------------------------------------------------\n# Main retrieval entry point\n# ---------------------------------------------------------------------------\n\ndef retrieve(\n    filename: str,\n    analysis: dict,\n    index: dict,\n    top_k: int = 3,\n) -> tuple[str, str]:\n    \"\"\"\n    Find the best guide context for this new CIP.\n\n    Returns (mode, context_text) where mode is \"prior_year\", \"similar\", or \"none\".\n    \"\"\"\n    # 1. Prior-year exact match \u2014 highest quality, use full guide\n    prior = find_prior_year(filename, index)\n    if prior:\n        excerpt = get_guide_excerpt(prior[\"guide_path\"], mode=\"full\")\n        print(f\"  Guide RAG: prior-year match \u2192 {prior['agency_id']}\")\n        return \"prior_year\", excerpt\n\n    # 2. Similar guides \u2014 use config constants from top matches\n    similar = find_similar(filename, analysis, index, top_k=top_k)\n    if not similar:\n        print(\"  Guide RAG: no matches found.\")\n        return \"none\", \"\"\n\n    parts = []\n    for g in similar:\n        excerpt = get_guide_excerpt(g[\"guide_path\"], mode=\"config\")\n        header = (\n            f\"--- Guide: {g['agency_id']} \"\n            f\"(layout={g['format_type']}, type={g['doc_type']}, years={g['years']}) ---\"\n        )\n        parts.append(f\"{header}\\n{excerpt}\")\n        print(f\"  Guide RAG: similar \u2192 {g['agency_id']}\")\n\n    return \"similar\", \"\\n\\n\".join(parts)\n", "runner": "\"\"\"\nSteps 4 & 5: Run the extraction script in a subprocess sandbox.\n\n- Step 4: single-project test (run + capture first 3 rows, show curator)\n- Step 5: full extraction \u2192 _final.csv\n\nThe script is executed via subprocess so that any errors are isolated.\nThe script's run() function must return list[dict].\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport subprocess\nimport sys\nimport tempfile\nimport textwrap\nfrom pathlib import Path\nfrom typing import Any\n\n\ndef _run_script(script_path: Path, capture_n: int | None = None) -> tuple[list[dict], str]:\n    \"\"\"\n    Execute the extraction script and return (rows, stderr).\n    The script must define run() returning list[dict].\n    \"\"\"\n    # We inject a small wrapper that calls run() and JSON-serialises the result\n    limit_code = f\"rows = rows[:{capture_n}]\" if capture_n else \"\"\n    wrapper = textwrap.dedent(f\"\"\"\nimport sys, json, importlib.util\nspec = importlib.util.spec_from_file_location(\"_cip_script\", r{str(script_path)!r})\nmod  = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(mod)\nrows = mod.run()\n{limit_code}\n# Stringify any non-serialisable values\nclean = [{{str(k): str(v) if not isinstance(v, (str,int,float,type(None))) else v\n           for k,v in r.items()}} for r in rows]\nprint(json.dumps(clean))\n\"\"\")\n\n    with tempfile.NamedTemporaryFile(mode=\"w\", suffix=\".py\", delete=False, encoding=\"utf-8\") as f:\n        f.write(wrapper)\n        tmp = Path(f.name)\n\n    try:\n        result = subprocess.run(\n            [sys.executable, str(tmp)],\n            capture_output=True,\n            text=True,\n            timeout=120,\n        )\n        stderr = result.stderr.strip()\n        if result.returncode != 0:\n            raise RuntimeError(f\"Script exited with code {result.returncode}:\\n{stderr}\")\n        rows = json.loads(result.stdout)\n        return rows, stderr\n    finally:\n        tmp.unlink(missing_ok=True)\n\n\ndef test_run(script_path: Path, n: int = 3) -> tuple[list[dict], str]:\n    \"\"\"Step 4: Run the script, return first n rows for curator review.\"\"\"\n    return _run_script(script_path, capture_n=n)\n\n\ndef full_run(script_path: Path) -> tuple[list[dict], str]:\n    \"\"\"Step 5: Run the script on all rows.\"\"\"\n    return _run_script(script_path, capture_n=None)\n\n\ndef save_final(rows: list[dict], out_path: Path) -> None:\n    \"\"\"Write rows to _final.csv using the standard schema column order.\"\"\"\n    from .schema import FINAL_COLS\n\n    # Build fieldnames: standard cols first, then any extras\n    all_keys = set()\n    for r in rows:\n        all_keys.update(r.keys())\n\n    fieldnames = [c for c in FINAL_COLS if c in all_keys]\n    extras = sorted(all_keys - set(FINAL_COLS))\n    fieldnames += extras\n\n    with open(out_path, \"w\", newline=\"\", encoding=\"utf-8\") as f:\n        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction=\"ignore\")\n        writer.writeheader()\n        writer.writerows(rows)\n", "qa": "\"\"\"\nStep 6: Exhaustive QA \u2014 source inventory vs extracted inventory.\n\nPer the WI \u00a76:\n- Build source inventory: ID + title + total amount for every project in the raw source\n- Build extract inventory: same from _final.csv rows\n- One-to-one matching: exact ID match, fuzzy title \u226585%, dollar tolerance $1 for amounts \u2265$1M\n- Report every discrepancy; NEVER auto-correct (WI \u00a76.6)\n\nTwo modes:\n  qa_totals()    \u2014 fast: compare aggregate sums only (for PDF-extracted data)\n  qa_full()      \u2014 thorough: one-to-one row matching (for tabular data)\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom difflib import SequenceMatcher\nfrom typing import Any\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef _to_num(v: Any) -> float:\n    if v is None:\n        return 0.0\n    try:\n        s = re.sub(r\"[,$\\s]\", \"\", str(v))\n        return float(s) if s else 0.0\n    except ValueError:\n        return 0.0\n\n\ndef _title_sim(a: str, b: str) -> float:\n    return SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio()\n\n\n# ---------------------------------------------------------------------------\n# Source inventory builder (for tabular files: CSV/Excel)\n# ---------------------------------------------------------------------------\n\ndef build_source_inventory(rows: list[dict], id_col: str, title_col: str, amount_col: str) -> list[dict]:\n    \"\"\"Build source inventory from raw rows.\"\"\"\n    inv = []\n    for i, r in enumerate(rows, 1):\n        inv.append({\n            \"index\":  i,\n            \"id\":     str(r.get(id_col, \"\")).strip(),\n            \"title\":  str(r.get(title_col, \"\")).strip(),\n            \"amount\": _to_num(r.get(amount_col, 0)),\n        })\n    return inv\n\n\ndef build_extract_inventory(rows: list[dict]) -> list[dict]:\n    \"\"\"Build extract inventory from _final.csv rows.\"\"\"\n    inv = []\n    for r in rows:\n        inv.append({\n            \"index\":  r.get(\"Project_Index\", \"\"),\n            \"id\":     str(r.get(\"Project_Number_Primary\", \"\")).strip(),\n            \"title\":  str(r.get(\"Project_Title\", \"\")).strip(),\n            \"amount\": _to_num(r.get(\"Total_Project_Budget\", 0)),\n        })\n    return inv\n\n\n# ---------------------------------------------------------------------------\n# Full one-to-one QA\n# ---------------------------------------------------------------------------\n\ndef qa_full(\n    source_inv: list[dict],\n    extract_inv: list[dict],\n    id_match: bool = True,\n    title_threshold: float = 0.85,\n    amount_tolerance: float = 1.0,\n    large_amount_threshold: float = 1_000_000,\n) -> dict:\n    \"\"\"\n    Match source_inv rows against extract_inv rows.\n    Returns a dict with: ok, warnings, errors, matches, unmatched_source, unmatched_extract\n    \"\"\"\n    result = {\n        \"ok\": [],\n        \"warnings\": [],\n        \"errors\": [],\n        \"matched_pairs\": [],\n        \"unmatched_source\": [],\n        \"unmatched_extract\": [],\n    }\n\n    used_extract = set()\n\n    for src in source_inv:\n        best_match = None\n        best_score = 0.0\n\n        for i, ext in enumerate(extract_inv):\n            if i in used_extract:\n                continue\n\n            # ID match (if IDs are present)\n            if id_match and src[\"id\"] and ext[\"id\"]:\n                if src[\"id\"].upper() == ext[\"id\"].upper():\n                    best_match, best_score = i, 1.0\n                    break\n\n            # Title fuzzy match\n            sim = _title_sim(src[\"title\"], ext[\"title\"])\n            if sim > best_score:\n                best_score, best_match = sim, i\n\n        if best_match is None or best_score < title_threshold:\n            result[\"errors\"].append({\n                \"type\": \"UNMATCHED_SOURCE\",\n                \"source\": src,\n                \"best_score\": best_score,\n            })\n            result[\"unmatched_source\"].append(src)\n            continue\n\n        used_extract.add(best_match)\n        ext = extract_inv[best_match]\n        result[\"matched_pairs\"].append({\"source\": src, \"extract\": ext, \"title_sim\": best_score})\n\n        # Amount check\n        s_amt, e_amt = src[\"amount\"], ext[\"amount\"]\n        threshold = amount_tolerance if max(s_amt, e_amt) >= large_amount_threshold else amount_tolerance\n        if abs(s_amt - e_amt) > threshold:\n            result[\"errors\"].append({\n                \"type\": \"AMOUNT_DIFF\",\n                \"source\": src,\n                \"extract\": ext,\n                \"diff\": e_amt - s_amt,\n            })\n        else:\n            result[\"ok\"].append(src[\"title\"][:60])\n\n    # Anything in extract that wasn't matched\n    for i, ext in enumerate(extract_inv):\n        if i not in used_extract:\n            result[\"unmatched_extract\"].append(ext)\n            result[\"warnings\"].append({\n                \"type\": \"EXTRA_EXTRACT\",\n                \"extract\": ext,\n            })\n\n    return result\n\n\n# ---------------------------------------------------------------------------\n# Totals-only QA (for PDF extracts or when no row-level source is available)\n# ---------------------------------------------------------------------------\n\ndef qa_totals(\n    expected: dict[str, float],\n    extracted_rows: list[dict],\n    tolerance: float = 1.0,\n) -> dict:\n    \"\"\"\n    Compare aggregate sums.\n    expected: {\"total\": X, \"2026\": Y, \"2027\": Z, ...}\n    \"\"\"\n    result = {\"ok\": [], \"errors\": [], \"warnings\": []}\n\n    # Compute extracted totals\n    actual: dict[str, float] = {}\n\n    # Grand total\n    total = sum(_to_num(r.get(\"Total_Project_Budget\", 0)) for r in extracted_rows)\n    actual[\"total\"] = total\n\n    # Year totals from Yearly_Costs_By_Category_JSON\n    for r in extracted_rows:\n        try:\n            yearly = json.loads(r.get(\"Yearly_Costs_By_Category_JSON\", \"{}\") or \"{}\")\n        except (json.JSONDecodeError, TypeError):\n            yearly = {}\n        for yr, amt in yearly.items():\n            actual[str(yr)] = actual.get(str(yr), 0.0) + _to_num(amt)\n\n    for key, exp_val in expected.items():\n        act_val = actual.get(str(key), 0.0)\n        diff = act_val - exp_val\n        label = f\"{key}: expected {exp_val:,.0f}, got {act_val:,.0f} (diff {diff:+,.0f})\"\n        if abs(diff) <= tolerance:\n            result[\"ok\"].append(label)\n        else:\n            result[\"errors\"].append({\"key\": key, \"expected\": exp_val, \"actual\": act_val, \"diff\": diff})\n\n    return result\n\n\n# ---------------------------------------------------------------------------\n# Report formatter\n# ---------------------------------------------------------------------------\n\ndef format_report(qa_result: dict) -> str:\n    lines = []\n\n    ok = qa_result.get(\"ok\", [])\n    errs = qa_result.get(\"errors\", [])\n    warns = qa_result.get(\"warnings\", [])\n\n    lines.append(f\"QA RESULT:  {len(ok)} OK  |  {len(errs)} ERR  |  {len(warns)} WARN\")\n\n    if errs:\n        lines.append(\"\\n--- ERRORS (must review) ---\")\n        for e in errs:\n            t = e.get(\"type\", \"ERROR\")\n            if t == \"AMOUNT_DIFF\":\n                src = e[\"source\"]\n                lines.append(\n                    f\"  DIFF  {src['id']:15s}  {src['title'][:40]:40s}  \"\n                    f\"src={e['source']['amount']:>15,.0f}  \"\n                    f\"ext={e['extract']['amount']:>15,.0f}  \"\n                    f\"diff={e['diff']:+,.0f}\"\n                )\n            elif t == \"UNMATCHED_SOURCE\":\n                src = e[\"source\"]\n                lines.append(f\"  UNMATCHED  {src['id']:15s}  {src['title'][:50]}  (best sim={e['best_score']:.0%})\")\n            else:\n                lines.append(f\"  {e}\")\n\n    if warns:\n        lines.append(\"\\n--- WARNINGS ---\")\n        for w in warns:\n            if isinstance(w, dict):\n                t = w.get(\"type\", \"WARN\")\n                if t == \"EXTRA_EXTRACT\":\n                    ext = w[\"extract\"]\n                    lines.append(f\"  EXTRA  {ext['id']:15s}  {ext['title'][:50]}\")\n            else:\n                lines.append(f\"  {w}\")\n\n    unmatched = qa_result.get(\"unmatched_source\", [])\n    if unmatched:\n        lines.append(f\"\\n  {len(unmatched)} source projects not found in extract.\")\n\n    return \"\\n\".join(lines)\n", "validate": "\"\"\"\nStep 7: Structured validation \u2014 schema checks, distributions, spot checks.\n\nRuns after Step 6 QA. Reports summary statistics and any structural issues.\nDoes NOT auto-fix anything (WI \u00a76.6).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom collections import Counter\nfrom typing import Any\n\nfrom .schema import FINAL_COLS, REQUIRED_COLS, gate_cell7, _to_num\n\n\ndef run(rows: list[dict], source_label: str = \"\") -> str:\n    \"\"\"\n    Full validation report. Returns formatted string.\n    Raises ValueError on fatal structural failures.\n    \"\"\"\n    lines = []\n    n = len(rows)\n    label = f\"[{source_label}] \" if source_label else \"\"\n\n    lines.append(f\"=== VALIDATION REPORT: {source_label} ===\")\n    lines.append(f\"Total rows: {n}\")\n\n    # --- Schema presence ---\n    cols_present = set(rows[0].keys()) if rows else set()\n    missing_required = REQUIRED_COLS - cols_present\n    if missing_required:\n        raise ValueError(f\"{label}Missing required columns: {missing_required}\")\n\n    missing_standard = set(FINAL_COLS) - cols_present\n    if missing_standard:\n        lines.append(f\"  Missing standard cols (non-fatal): {sorted(missing_standard)}\")\n    else:\n        lines.append(\"  All standard columns present.\")\n\n    # --- Run Cell 7 gate ---\n    try:\n        warnings = gate_cell7(rows, source_label)\n        for w in warnings:\n            lines.append(f\"  WARN: {w}\")\n    except ValueError as e:\n        lines.append(f\"  FATAL: {e}\")\n        raise\n\n    # --- Budget distribution ---\n    budgets = [_to_num(r.get(\"Total_Project_Budget\", 0)) for r in rows]\n    budgets_nonzero = [b for b in budgets if b > 0]\n    grand_total = sum(budgets)\n    lines.append(f\"\\nBudget summary:\")\n    lines.append(f\"  Grand total:     ${grand_total:>20,.0f}\")\n    lines.append(f\"  Non-zero rows:   {len(budgets_nonzero):>6} / {n}\")\n    if budgets_nonzero:\n        lines.append(f\"  Min (non-zero):  ${min(budgets_nonzero):>20,.0f}\")\n        lines.append(f\"  Max:             ${max(budgets_nonzero):>20,.0f}\")\n        lines.append(f\"  Mean:            ${sum(budgets_nonzero)/len(budgets_nonzero):>20,.0f}\")\n\n    # --- Department distribution ---\n    depts = [str(r.get(\"Client_Department\", \"\")).strip() for r in rows]\n    dept_counter = Counter(d for d in depts if d)\n    if dept_counter:\n        lines.append(f\"\\nDepartments ({len(dept_counter)} unique):\")\n        for dept, cnt in dept_counter.most_common(10):\n            lines.append(f\"  {cnt:4d}  {dept[:60]}\")\n        if len(dept_counter) > 10:\n            lines.append(f\"  ... and {len(dept_counter)-10} more\")\n\n    # --- Fund types ---\n    all_funds: Counter = Counter()\n    for r in rows:\n        try:\n            names = json.loads(r.get(\"Fund_Names_JSON\", \"[]\") or \"[]\")\n            budgets_j = json.loads(r.get(\"Fund_Budgets_JSON\", \"[]\") or \"[]\")\n            for name, amt in zip(names, budgets_j):\n                all_funds[name] += _to_num(amt)\n        except (json.JSONDecodeError, TypeError):\n            pass\n    if all_funds:\n        lines.append(f\"\\nFund breakdown (grand total by fund):\")\n        for fund, amt in sorted(all_funds.items(), key=lambda x: -x[1]):\n            lines.append(f\"  {fund:<30s}  ${amt:>20,.0f}\")\n\n    # --- Year coverage ---\n    year_totals: dict[str, float] = {}\n    for r in rows:\n        try:\n            yearly = json.loads(r.get(\"Yearly_Costs_By_Category_JSON\", \"{}\") or \"{}\")\n            if isinstance(yearly, dict):\n                for yr, amt in yearly.items():\n                    year_totals[str(yr)] = year_totals.get(str(yr), 0) + _to_num(amt)\n        except (json.JSONDecodeError, TypeError):\n            pass\n    if year_totals:\n        lines.append(f\"\\nYearly totals:\")\n        for yr in sorted(year_totals):\n            lines.append(f\"  {yr}:  ${year_totals[yr]:>20,.0f}\")\n\n    # --- Blank field rates ---\n    lines.append(f\"\\nField fill rates:\")\n    for col in FINAL_COLS:\n        filled = sum(1 for r in rows if str(r.get(col, \"\")).strip())\n        pct = filled / n * 100 if n else 0\n        flag = \"  \" if pct >= 80 else \"! \" if pct >= 50 else \"!!\"\n        lines.append(f\"  {flag}{col:<40s}  {filled:5d}/{n}  ({pct:.0f}%)\")\n\n    lines.append(\"\\n=== END VALIDATION ===\")\n    return \"\\n\".join(lines)\n"}

os.makedirs('/tmp/cip_tools', exist_ok=True)
with open('/tmp/cip_tools/__init__.py', 'w') as _f:
    _f.write('"""cip_tools."""\n')
for _name, _code in _CIP_MODULES.items():
    with open(f'/tmp/cip_tools/{_name}.py', 'w', encoding='utf-8') as _f:
        _f.write(_code)
if '/tmp' not in sys.path:
    sys.path.insert(0, '/tmp')

# Create base extract folder structure on Drive
_base = '/content/drive/Shareddrives/0_cip_data/extract'
for _folder in [_base, f'{_base}/1_inbox']:
    os.makedirs(_folder, exist_ok=True)
print("Dependencies ready. Extract folders created.")


In [ ]:
#@title Step 0b — Configuration
import os, datetime, ipywidgets as _w
from IPython.display import display, Markdown

_EXTRACT_ROOT = "/content/drive/Shareddrives/0_cip_data/extract"
_inbox = f"{_EXTRACT_ROOT}/1_inbox"

_files = sorted(
    f for f in os.listdir(_inbox)
    if os.path.isfile(f'{_inbox}/{f}') and not f.startswith('.')
)
if not _files:
    display(Markdown("**No files in `1_inbox/`** — upload a source file first."))
else:
    _picker = _w.Dropdown(options=_files, description='Source file:',
                          layout=_w.Layout(width='700px'))
    _out    = _w.Output()

    def _pick(change=None):
        global SOURCE_FILE, AGENCY_ID, OUT_DIR, SCRIPTS_DIR
        SOURCE_FILE = f"{_inbox}/{_picker.value}"
        AGENCY_ID   = os.path.splitext(_picker.value)[0]
        _today      = datetime.date.today().strftime('%Y %m %d')
        OUT_DIR     = f"{_EXTRACT_ROOT}/{_today} - {AGENCY_ID}"
        SCRIPTS_DIR = OUT_DIR
        os.makedirs(OUT_DIR, exist_ok=True)
        _out.clear_output()
        with _out:
            display(Markdown(
                f"**Agency ID:** `{AGENCY_ID}`  \n"
                f"**Output folder:** `{OUT_DIR}`"
            ))

    _picker.observe(_pick, names='value')
    display(_w.VBox([_picker, _out]))
    _pick()


In [ ]:
#@title Step 0c — Guide Index (run once per session)
from cip_tools import guide_rag
from IPython.display import display, Markdown
import os

_GUIDE_ROOT  = "/content/drive/Shareddrives/0_cip_data/2026"
_INDEX_PATH  = "/content/drive/Shareddrives/0_cip_data/extract/_guide_index.json"

if os.path.exists(_INDEX_PATH):
    _GUIDE_INDEX = guide_rag.load_index(_INDEX_PATH)
    display(Markdown(
        f"**Guide index loaded:** {len(_GUIDE_INDEX['guides'])} guides "
        f"(built {_GUIDE_INDEX['built_at'][:10]})"
    ))
else:
    print("Building guide index (first run — scans all *_guide.md files)...")
    _GUIDE_INDEX = guide_rag.build_index(_GUIDE_ROOT, _INDEX_PATH)
    display(Markdown(f"**Guide index built:** {len(_GUIDE_INDEX['guides'])} guides indexed."))


---
## Extraction Steps

Run each cell, review, then continue.

In [ ]:
#@title Steps 1-2 — Load File & Analyze Structure  (Claude)
from cip_tools import ingest, schema, understand
from IPython.display import display, Markdown

# --- Step 1: Load & Normalize ---
print(f"Loading: {SOURCE_FILE}")
_raw_text, _sample_rows, _metadata = ingest.load(SOURCE_FILE)

_c3_warns = []
_c3_fatal = None
try:
    _c3_warns = schema.gate_cell3(_sample_rows, AGENCY_ID, fmt=_metadata.get('format'))
except ValueError as _e:
    _c3_fatal = str(_e)

fmt   = _metadata.get('format', '?')
ncols = _metadata.get('n_cols', '?')
nrows = _metadata.get('estimated_rows', '?')

if _c3_fatal:
    display(Markdown(f"## INGESTION FAILURE\n\n**{_c3_fatal}**\n\nStop and check the source file."))
    raise SystemExit(_c3_fatal)

display(Markdown(
    f"## File Loaded\n\n"
    f"| | |\n|--|--|\n"
    f"| Format | `{fmt}` |\n"
    f"| Columns | {ncols} |\n"
    f"| Rows (est.) | {nrows} |\n"
))
if _c3_warns:
    display(Markdown("**Warnings:**\n" + "\n".join(f"- {w}" for w in _c3_warns)))
if _sample_rows:
    cols = list(_sample_rows[0].keys())
    col_list = "\n".join(f"{i+1}. `{c}`" for i, c in enumerate(cols[:25]))
    extra = f"\n*...and {len(cols)-25} more*" if len(cols) > 25 else ""
    display(Markdown(f"**Columns detected:**\n{col_list}{extra}"))

# --- Step 2: Analyze Structure (Claude) ---
print("\nAsking Claude to analyze structure...")
_analysis = understand.analyze(
    os.path.basename(SOURCE_FILE), _raw_text, _sample_rows, _metadata
)
_analysis_summary = understand.summarize(_analysis)

display(Markdown(
    f"## Structure Analysis\n\n"
    f"```\n{_analysis_summary}\n```"
))
if _analysis.get('quirks'):
    display(Markdown("**Quirks noted:**\n" + "\n".join(f"- {q}" for q in _analysis['quirks'])))
if _analysis.get('notes'):
    display(Markdown(f"**Notes:** {_analysis['notes']}"))

display(Markdown(
    "---\n"
    "*Review the above. If it looks correct, run the next cell.  \n"
    "If something is wrong, stop and contact your supervisor.*"
))


In [ ]:
#@title Step 3 — Generate Extraction Script  (Claude)
from cip_tools import design, guide_rag
from IPython.display import display, Markdown, Code
from pathlib import Path

# Retrieve guide context (prior-year if available, else similar)
_guide_mode, _guide_context = guide_rag.retrieve(
    os.path.basename(SOURCE_FILE), _analysis, _GUIDE_INDEX,
)
if _guide_mode == "prior_year":
    display(Markdown("**Guide RAG:** Prior-year guide found — using as primary template."))
elif _guide_mode == "similar":
    display(Markdown("**Guide RAG:** Using similar past extractions as reference."))
else:
    display(Markdown("*Guide RAG: no matches — generating script from scratch.*"))

print("Asking Claude to write extraction script...")
_script_code = design.write_script(
    os.path.basename(SOURCE_FILE), _analysis, _sample_rows,
    full_path=SOURCE_FILE,
    metadata=_metadata,
    guide_context=_guide_context,
    guide_mode=_guide_mode,
)

_script_path = Path(SCRIPTS_DIR) / f"{AGENCY_ID}_extract.py"
_script_path.write_text(_script_code, encoding='utf-8')

nlines = len(_script_code.splitlines())
display(Markdown(f"**Script generated:** {nlines} lines — saved to `{_script_path}`"))
display(Code(_script_code[:4000], language='python'))
if nlines > 60:
    display(Markdown(f"*({nlines - 60} more lines — open the file on Drive to view/edit all)*"))

display(Markdown(
    "---\n"
    "**Review the script above.**  \n"
    "If it looks correct, run the next cell.  \n"
    "If it needs changes, open the `.py` file on Drive, edit and save it, then run the next cell."
))


In [ ]:
#@title Step 4 — Test Run (3 rows)
from cip_tools import runner
from IPython.display import display, Markdown

# Re-read script from Drive in case curator edited it
_script_code = _script_path.read_text(encoding='utf-8')
_script_path.write_text(_script_code, encoding='utf-8')

print("Test run (first 3 rows)...")
try:
    _test_rows, _test_stderr = runner.test_run(_script_path, n=3)
except RuntimeError as _err:
    display(Markdown(f"## Script Error\n\n```\n{_err}\n```\n\nFix the script and re-run this cell."))
    raise

if _test_stderr:
    display(Markdown(f"**Script warnings:**\n```\n{_test_stderr[:400]}\n```"))

display(Markdown(f"## Test Results ({len(_test_rows)} rows extracted)"))
for _r in _test_rows:
    _budget = float(_r.get('Total_Project_Budget') or 0)
    _yearly = _r.get('Yearly_Costs_By_Category_JSON', '{}')
    _funds  = _r.get('Fund_Names_JSON', '[]')
    display(Markdown(
        f"**[{_r.get('Project_Index')}]** {_r.get('Project_Title', '?')}  \n"
        f"- ID: `{_r.get('Project_Number_Primary', '?')}`  "
        f"Dept: `{_r.get('Client_Department', '?')}`  \n"
        f"- Total budget: **${_budget:,.0f}**  \n"
        f"- Yearly: `{_yearly}`  \n"
        f"- Funds: `{_funds}`"
    ))

display(Markdown(
    "---\n"
    "*Verify these rows against your source document.*  \n"
    "*If correct, run the next cell. If wrong, edit the script and re-run this cell.*"
))


In [ ]:
#@title Steps 5-8 — Full Extraction, Validate & Save
from cip_tools import runner, validate, design
from IPython.display import display, Markdown
from pathlib import Path
import datetime

# Re-read script from Drive in case curator edited it after the test run
_script_code = _script_path.read_text(encoding='utf-8')

# --- Step 5: Full Extraction ---
print("Running full extraction...")
_full_rows, _full_stderr = runner.full_run(_script_path)

if _full_stderr:
    display(Markdown(f"**Script warnings:**\n```\n{_full_stderr[:400]}\n```"))

_grand_total = sum(float(r.get('Total_Project_Budget') or 0) for r in _full_rows)

display(Markdown(
    f"## Extraction Complete\n\n"
    f"| | |\n|--|--|\n"
    f"| Projects extracted | **{len(_full_rows)}** |\n"
    f"| Grand total | **${_grand_total:,.0f}** |\n"
))

_published = _analysis.get('published_grand_total')
if _published:
    _diff = _grand_total - _published
    _pct  = _diff / _published * 100
    _ok   = abs(_pct) < 1.0
    display(Markdown(
        f"### Total Check: {'PASS ✓' if _ok else 'FAIL ✗'}\n\n"
        f"Published: **${_published:,.0f}** | "
        f"Extracted: **${_grand_total:,.0f}** | "
        f"Diff: **${_diff:+,.0f}** ({_pct:+.2f}%)"
    ))
else:
    display(Markdown(
        "_Published grand total not found in document — "
        "verify extracted total manually against source._"
    ))

# --- Step 6-7: QA & Validation ---
print("\nRunning QA and validation...")
_val_ok = True
try:
    _val_report = validate.run(_full_rows, source_label=AGENCY_ID)
    display(Markdown(f"```\n{_val_report}\n```"))
except ValueError as _e:
    _val_ok = False
    display(Markdown(f"## VALIDATION FAILURE\n\n**{_e}**\n\nFix the script and re-run this cell."))
    raise

display(Markdown(
    "**All discrepancies above require human review — nothing is auto-corrected.**  \n"
    "If errors are found: edit the script, re-run Step 4, then re-run this cell."
))

# --- Step 8: Save (only after validation passes) ---
_final_path = Path(OUT_DIR) / f"{AGENCY_ID}_final.csv"

# Version sweep: rename existing _final.csv before overwriting
if _final_path.exists():
    _ts = datetime.datetime.now().strftime('%H%M%S')
    _prev_path = Path(OUT_DIR) / f"{AGENCY_ID}_final_prev_{_ts}.csv"
    _final_path.rename(_prev_path)
    display(Markdown(f"*Previous output archived as `{_prev_path.name}`*"))

runner.save_final(_full_rows, _final_path)

_guide_content = design.make_guide(
    agency_id=AGENCY_ID,
    filename=os.path.basename(SOURCE_FILE),
    analysis=_analysis,
    script=_script_code,
    analysis_summary=_analysis_summary,
)
_guide_path = Path(OUT_DIR) / f"{AGENCY_ID}_guide.md"
_guide_path.write_text(_guide_content, encoding='utf-8')

display(Markdown(
    f"## Done\n\n"
    f"Files saved to `{OUT_DIR}`:\n\n"
    f"| File | Description |\n|------|-------------|\n"
    f"| `{AGENCY_ID}_extract.py` | Extraction script — reuse next year |\n"
    f"| `{AGENCY_ID}_final.csv` | Standard output ({len(_full_rows)} rows) |\n"
    f"| `{AGENCY_ID}_guide.md` | Curator guide with QA checklist |\n"
))
